In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

data = sns.load_dataset("titanic")
print("Shape:", data.shape)
data.head()

Shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [2]:
print("Duplicated rows:", data.duplicated().sum())
data = data.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", data.shape)

Duplicated rows: 107
Shape after removing duplicates: (784, 15)


In [3]:
missing_percentage = data.isnull().mean() * 100
print(missing_percentage.sort_values(ascending=False).round(2))

missing_threshold = 70
cols_to_remove = missing_percentage[missing_percentage >= missing_threshold].index
print("Columns removed:", list(cols_to_remove))
data = data.drop(columns=cols_to_remove)

deck           74.23
age            13.52
embarked        0.26
embark_town     0.26
sex             0.00
pclass          0.00
survived        0.00
fare            0.00
parch           0.00
sibsp           0.00
class           0.00
adult_male      0.00
who             0.00
alive           0.00
alone           0.00
dtype: float64
Columns removed: ['deck']


In [4]:
complete_data = data.dropna()
age_data = data.dropna(subset=["age"])

print("Rows after removing every incomplete row:", len(complete_data))
print("Rows after removing missing age:", len(age_data))

Rows after removing every incomplete row: 676
Rows after removing missing age: 678


In [5]:
num_cols = data.select_dtypes(include=np.number).columns
cat_cols = data.select_dtypes(include=["object", "category"]).columns

for col in num_cols:
    data[col] = data[col].fillna(data[col].median())

for col in cat_cols:
    if data[col].isnull().any():
        data[col] = data[col].fillna(data[col].mode()[0])

print("Remaining missing values:", int(data.isnull().sum().sum()))

Remaining missing values: 0


In [6]:
q1 = data["fare"].quantile(0.25)
q3 = data["fare"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = data[(data["fare"] < lower) | (data["fare"] > upper)]
print("Fare outliers before capping:", len(outliers))

data["fare"] = data["fare"].clip(lower, upper)

Fare outliers before capping: 102


In [7]:
data[num_cols] = StandardScaler().fit_transform(data[num_cols])
data[num_cols].head()

,survived,pclass,age,sibsp,parch,fare
0,-0.837049,0.885158,-0.557360,0.484009,-0.497157,-0.845924
1,1.194673,-1.455362,0.608298,0.484009,-0.497157,1.969018
2,1.194673,0.885158,-0.265945,-0.530599,-0.497157,-0.816251
3,1.194673,-1.455362,0.389737,0.484009,-0.497157,1.169669
4,-0.837049,0.885158,0.389737,-0.530599,-0.497157,-0.810756


In [8]:
data = pd.get_dummies(data, columns=["embarked"], dtype=int)
print("Remaining missing values:", int(data.isnull().sum().sum()))
data.head()

Remaining missing values: 0


,survived,pclass,sex,age,sibsp,parch,fare,class,who,adult_male,embark_town,alive,alone,embarked_C,embarked_Q,embarked_S
0,-0.837049,0.885158,male,-0.557360,0.484009,-0.497157,-0.845924,Third,man,True,Southampton,no,False,0,0,1
1,1.194673,-1.455362,female,0.608298,0.484009,-0.497157,1.969018,First,woman,False,Cherbourg,yes,False,1,0,0
2,1.194673,0.885158,female,-0.265945,-0.530599,-0.497157,-0.816251,Third,woman,False,Southampton,yes,True,0,0,1
3,1.194673,-1.455362,female,0.389737,0.484009,-0.497157,1.169669,First,woman,False,Southampton,yes,False,0,0,1
4,-0.837049,0.885158,male,0.389737,-0.530599,-0.497157,-0.810756,Third,man,True,Southampton,no,True,0,0,1


In [10]:
if "embarked" in data.columns:
    data = pd.get_dummies(data, columns=["embarked"], dtype=int)
print("Remaining missing values:", int(data.isnull().sum().sum()))
data.head()

Remaining missing values: 0


,survived,pclass,sex,age,sibsp,parch,fare,class,who,adult_male,embark_town,alive,alone,embarked_C,embarked_Q,embarked_S
0,-0.837049,0.885158,male,-0.557360,0.484009,-0.497157,-0.845924,Third,man,True,Southampton,no,False,0,0,1
1,1.194673,-1.455362,female,0.608298,0.484009,-0.497157,1.969018,First,woman,False,Cherbourg,yes,False,1,0,0
2,1.194673,0.885158,female,-0.265945,-0.530599,-0.497157,-0.816251,Third,woman,False,Southampton,yes,True,0,0,1
3,1.194673,-1.455362,female,0.389737,0.484009,-0.497157,1.169669,First,woman,False,Southampton,yes,False,0,0,1
4,-0.837049,0.885158,male,0.389737,-0.530599,-0.497157,-0.810756,Third,man,True,Southampton,no,True,0,0,1
